# Data Exploration

This notebook details steps taken to investigate and understand the Food Standards Agency (FSA) API, which details Food Hygiene Recognition Scheme (FHRS) ratings for businesses in the UK.  

[API Documentation Link](https://api.ratings.food.gov.uk/Help).  

This is being completed as a first step to examine the response / output of the API, in order to plan and shape the rest of the project. 

_Note: the data used is pulled from a live connection to the FSA database. This is updated frequently, so values (e.g. establishment count, ratings and ratings distribution) will change slighly over time. Quoted values in explanatory text are from the a pull on 23/08/26 and were those used to inform approaches used etc.._

### Initial Test Call - Check Status

The first check is to see if a status `200` is returned from a basic 'get' call.

In [1]:
import requests

BASE_URL = "https://api.ratings.food.gov.uk"
HEADERS = {"x-api-version": "2"} # this header is required, otherwise silently fails (and returns nothing)

response = requests.get(f"{BASE_URL}/Authorities", headers=HEADERS) # hit Authorities endpoint
    # https://api.ratings.food.gov.uk/Help/Api/GET-Authorities-pageNumber-pageSize
    
print(response.status_code)

200


-> An HTTP response status code of `200` ('OK') confirms a successful call.

### Getting London-specific Authority IDs

FHRS ratings are issued by local authorities. As this project focuses on London businesses only, queries will need to be scoped by London authority IDs.

I therefore need to pull out the relevant authority IDs from the `/Authorities` endpoint response above.   

In [2]:
authorities = response.json()["authorities"]
  # authorities is an array of objects, with each element being a new authority

print(len(authorities)) # how many authorities are returned from the initial call

print(authorities[0]) # inspect a single authority object

363
{'LocalAuthorityId': 197, 'LocalAuthorityIdCode': '760', 'Name': 'Aberdeen City', 'FriendlyName': 'aberdeen-city', 'Url': 'http://www.aberdeencity.gov.uk', 'SchemeUrl': '', 'Email': 'commercial@aberdeencity.gov.uk', 'RegionName': 'Scotland', 'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS760en-GB.xml', 'FileNameWelsh': None, 'EstablishmentCount': 2202, 'CreationDate': '2010-08-17T15:30:24.87', 'LastPublishedDate': '2026-08-23T00:39:02.233', 'SchemeType': 2, 'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/197'}]}


-> This returns 363 results - this is roughly in line with expectations, as there are approx. 380 total LAs in the UK.  

An example element looks as follows:   

```
{
    'LocalAuthorityId': 197, 
    'LocalAuthorityIdCode': '760', 
    'Name': 'Aberdeen City', 
    'FriendlyName': 'aberdeen-city', 
    'Url': 'http://www.aberdeencity.gov.uk', 
    'SchemeUrl': '', 
    'Email': 'commercial@aberdeencity.gov.uk', 
    'RegionName': 'Scotland', 
    'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS760en-GB.xml', 
    'FileNameWelsh': None, 
    'EstablishmentCount': 2202, 
    'CreationDate': '2010-08-17T15:30:24.87', 
    'LastPublishedDate': '2026-08-21T00:40:33.257', 
    'SchemeType': 2, 
    'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/197'}]
}
```

The `RegionName` key may be useful in filtering to London only ... assuming there is a `'London'` Region Name.  

In [3]:
# store a list of authorities, from the existing authorities list, where the authority is in the London region
london_authorities = [a for a in authorities if a["RegionName"] == "London"]

print(len(london_authorities)) # see how many return
print(london_authorities[0]) # see what one of the returned objects looks like

33
{'LocalAuthorityId': 88, 'LocalAuthorityIdCode': '501', 'Name': 'Barking and Dagenham', 'FriendlyName': 'barking-and-dagenham', 'Url': 'http://www.lbbd.gov.uk/Pages/Home.aspx', 'SchemeUrl': '', 'Email': 'foodsafety@lbbd.gov.uk', 'RegionName': 'London', 'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS501en-GB.xml', 'FileNameWelsh': None, 'EstablishmentCount': 1459, 'CreationDate': '2010-08-17T15:30:24.87', 'LastPublishedDate': '2026-08-23T00:32:35.28', 'SchemeType': 1, 'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/88'}]}


-> Filtering by a `RegionName` of `London` worked ok.  

Crucially, the authority count is now 33 - matching the 33 London boroughs (each being a Local Authority).  

Barking and Dagenham is the Local Authority returned, which (encouragingly), is indeed in London.  

`SchemeType` should be `1` for all London LAs, with a 0-5 star scale (note that Aberdeen City above uses scheme 2 instead).  

In [4]:
# get unique SchemeType values
scheme_types = set(a["SchemeType"] for a in london_authorities) 
print(scheme_types)

{1}


-> A return of `{1}` confirms that all 33 boroughs are using the same 0-5* scheme.

With the number of London LA results and a consistent Scheme Type confirmed, I can move on from 'Authority-level' checks, and on to establishment records.  

### Establishment-level checks

Now that authority checks have been completed, and results can reliably be scoped to London only, I am now going to look at establishment records.  

The [Establishments endpoint documentation](https://api.ratings.food.gov.uk/Help/Api/GET-Establishments_name_address_longitude_latitude_maxDistanceLimit_businessTypeId_schemeTypeKey_ratingKey_ratingOperatorKey_localAuthorityId_countryId_sortOptionKey_pageNumber_pageSize) confirms that a search parameter of `localAuthorityID={localAuthorityId}` can be used to filter returned results.  

Using Barking & Dagenham's `LocalAuthorityId` of `88` (_notably, not_ `LocalAuthorityIdCode`):

In [5]:
# `requests` uses `params=params` to build a URL query string
# ... which is certainly better than hand-writing "?localAuthorityId=88&..." and so on
params = {"localAuthorityId": 88, "pageSize": 5000} # pageSize of 5000 should cover all establishments in one go
    # proper pagination can (and should) be used when pulling all 33 boroughs

response = requests.get(f"{BASE_URL}/Establishments", headers=HEADERS, params=params, timeout=15)
establishments = response.json()["establishments"]

print(len(establishments))
print(establishments[0])

1459
{'AddressLine1': '', 'AddressLine2': '309 Wood Lane', 'AddressLine3': '', 'AddressLine4': 'Dagenham', 'BusinessName': '5 Elms Cafe', 'BusinessType': 'Restaurant/Cafe/Canteen', 'BusinessTypeID': 1, 'ChangesByServerID': 0, 'Distance': None, 'FHRSID': 1714830, 'LocalAuthorityBusinessID': '81398', 'LocalAuthorityCode': '501', 'LocalAuthorityEmailAddress': 'foodsafety@lbbd.gov.uk', 'LocalAuthorityName': 'Barking and Dagenham', 'LocalAuthorityWebSite': 'http://www.lbbd.gov.uk/Pages/Home.aspx', 'NewRatingPending': False, 'Phone': '', 'PostCode': 'RM8 3NH', 'RatingDate': '2024-07-13T00:00:00', 'RatingKey': 'fhrs_5_en-gb', 'RatingValue': '5', 'RightToReply': '', 'SchemeType': 'FHRS', 'geocode': {'longitude': '0.142421', 'latitude': '51.554493'}, 'scores': {'Hygiene': 5, 'Structural': 5, 'ConfidenceInManagement': 5}}


-> 1459 results seems about right in terms of establishments for a single borough. 

An establishment result looks like so:

```
{
  "AddressLine1": "",
  "AddressLine2": "309 Wood Lane",
  "AddressLine3": "",
  "AddressLine4": "Dagenham",
  "BusinessName": "5 Elms Cafe",
  "BusinessType": "Restaurant/Cafe/Canteen",
  "BusinessTypeID": 1,
  "ChangesByServerID": 0,
  "Distance": null,
  "FHRSID": 1714830,
  "LocalAuthorityBusinessID": "81398",
  "LocalAuthorityCode": "501",
  "LocalAuthorityEmailAddress": "foodsafety@lbbd.gov.uk",
  "LocalAuthorityName": "Barking and Dagenham",
  "LocalAuthorityWebSite": "http://www.lbbd.gov.uk/Pages/Home.aspx",
  "NewRatingPending": false,
  "Phone": "",
  "PostCode": "RM8 3NH",
  "RatingDate": "2024-07-13T00:00:00",
  "RatingKey": "fhrs_5_en-gb",
  "RatingValue": "5",
  "RightToReply": "",
  "SchemeType": "FHRS",
  "geocode": {
    "longitude": "0.142421",
    "latitude": "51.554493"
  },
  "scores": {
    "Hygiene": 5,
    "Structural": 5,
    "ConfidenceInManagement": 5
  }
}
```

Some key fields include:
- `BusinessName` - useful for labelling output
- `BusinessType` / `BusinessTypeId` - may be helpful for segmentation if suitably clean/structured
- `PostCode` - as a fallback if lat/long doesn't match (past experience tells me that FSA location data can be janky or unreliable at times)
- `RatingKey` / `RatingValue` / `RatingDate` - for obvious reasons. _Notably, these are singular_
- `geocode` - object contains latitude and longitude, supported by Tableau maps for easy plotting
- `scores` - object contains category scores (_Hygiene, Structural, Confidence in Management_)

One key takeaway here is that there is no 'previous rating' or equivalent, and nothing to suggest that history is available through this endpoint.  

With this in mind, it looks likely that the data I can build my project from is a 'snapshot' only. Looking at past ratings as a predictor is probably a no-go.  

### Ratings Distribution Checks

With the `/Establishments` endpoint returning useable results, scoped to specific Local Authorities, the next thing I am moving on to is the distribution of results.  

Part of my early thoughts in terms of what the project should cover was along the lines of predicting a 'FHRS fail'. 

The FSA has no specific definition of a 'fail', although ratings of 2* or below mean "major improvement necessary". In many businesses, however - my previous role included - anything less than 5* was considered a brand-damaging result, even if a 4* isn't _necessarily_ an indicator of serious food safety issues.  

Something I want to check is whether there are enough establishments represented in the 0-2 star range, and again in the 0-4 range - if this is too small (or big) a slice of the data, a model is unlikely to be able to learn from it. 

In [6]:
from collections import Counter

rating_counts = Counter(e["RatingValue"] for e in establishments)

for rating, count in sorted(rating_counts.items()):
    rating_f = rating + '* rating' if len(rating) == 1 else rating 
    print(f"{rating_f} : {count} instances")

0* rating : 4 instances
1* rating : 71 instances
2* rating : 27 instances
3* rating : 156 instances
4* rating : 272 instances
5* rating : 666 instances
AwaitingInspection : 232 instances
Exempt : 31 instances


-> 8 unique ratings returned, 0-5 and 'AwaitingInspection' / 'Exempt'.  

- ~8.6% of results sit in the 0-2* range - would likely need some balancing, but more than enough to work with
- ~44% of results are 0-4* - no balancing needed.  

These results are for Barking & Daghenham only, so I now need to check that a similar split is seen London-wide (and that there are no additional categories creeping in). I'll therefore pull all London results now too:

In [7]:
import time

# target list to hold all-London results (c. 81k)
all_establishments = []

# fetch all establishments for an authority, using pagination
    # approach added in as Westminster exceeds 5000 establishments
def fetch_all_establishments(authority_id, page_size=5000): # default set to 5000 as previously works ok
    all_records = [] # target list to hold all results from a request
    page_number = 1

    while True:
        params = {"localAuthorityId": authority_id, "pageSize": page_size, "pageNumber": page_number}
        response = requests.get(f"{BASE_URL}/Establishments", headers=HEADERS, params=params, timeout=15)
        data = response.json()

        all_records.extend(data["establishments"]) # extend used rather than append
            # as this ensures a flat list

        # total pages count for a call is returned in the response metadata
        total_pages = data["meta"]["totalPages"]
        if page_number >= total_pages:
            break

        page_number += 1 # increment on each pass
        time.sleep(0.5) # build in a delay to avoid hammering the endpoint
            # it's polite and also (hopefully) prevents rate limiting

    return all_records

# call API for each LA, adding each (flattened) list of establishments into the main (also flat!) list
for authority in london_authorities:
    borough_establishments = fetch_all_establishments(authority["LocalAuthorityId"])
    all_establishments.extend(borough_establishments) # see above
    print(f"{authority['Name']}: {len(borough_establishments)} establishments")

print(f"\nTotal: {len(all_establishments)}")

Barking and Dagenham: 1459 establishments
Barnet: 2859 establishments
Bexley: 1770 establishments
Brent: 2493 establishments
Bromley: 2475 establishments
Camden: 4260 establishments
City of London Corporation: 1840 establishments
Croydon: 3158 establishments
Ealing: 3733 establishments
Enfield: 2393 establishments
Greenwich: 2391 establishments
Hackney: 2448 establishments
Hammersmith and Fulham: 2016 establishments
Haringey: 1940 establishments
Harrow: 1903 establishments
Havering: 2021 establishments
Hillingdon: 2229 establishments
Hounslow: 2337 establishments
Islington: 2591 establishments
Kensington and Chelsea: 1998 establishments
Kingston-Upon-Thames: 1430 establishments
Lambeth: 2644 establishments
Lewisham: 2482 establishments
Merton: 1566 establishments
Newham: 2659 establishments
Redbridge: 2158 establishments
Richmond-Upon-Thames: 1578 establishments
Southwark: 3299 establishments
Sutton: 1329 establishments
Tower Hamlets: 3171 establishments
Waltham Forest: 1978 establishm

-> returns 81,217 establishments from the 33 boroughs.  

My initial approach didn't account for Westminster having over 5,000 results: `fetch_all_establishments` handles a paginated approach and helps ensure rate limiter isn't triggered (as warned by the documentation).  

Just as a sense check to see that the results roughly match the authority-level count totals:

In [8]:
expected_total = sum(a["EstablishmentCount"] for a in london_authorities) # totals from authority-level endpoint
print(f"Expected (from authority records): {expected_total}")
print(f"Actual (pulled): {len(all_establishments)}") # total pulled

Expected (from authority records): 81217
Actual (pulled): 81217


-> 81,217 for both.  

Which is good news - an exact match - although a minor drift/discrepancy would not have been major cause for alarm.  

In terms of distribution:

In [9]:
rating_counts = Counter(e["RatingValue"] for e in all_establishments)

for rating, count in sorted(rating_counts.items()):
    rating_f = rating + '* rating' if len(rating) == 1 else rating 
    print(f"{rating_f} : {count} instances")

0* rating : 300 instances
1* rating : 1676 instances
2* rating : 1997 instances
3* rating : 5968 instances
4* rating : 12558 instances
5* rating : 48503 instances
AwaitingInspection : 7492 instances
AwaitingPublication : 8 instances
Exempt : 2715 instances


-> one new rating, `AwaitingPublication` has appeared, but this seems self-explanatory and only has 8 instances.

In terms of the impact this distrubution has on either a three-tier predictive model (i.e. pass, fail, critical fail) or one of the two binary (0-2* or 0-4*) fails:

In [10]:
numeric_ratings = {"0", "1", "2", "3", "4", "5"} # whitelist (numeric) values

# pull out only 'gradable' establishments (i.e., not exempt, awaiting inspection etc.)
gradable = [e for e in all_establishments if e["RatingValue"] in numeric_ratings]
total = len(gradable) # how many pass the filter
print(f"Gradable establishments: {total}")

# calculate percentage value, return a formatted string
def pct(n, total):
    return f"{n} ({n/total:.1%})" # 1 decimal place, as a pct (i.e n *100 with '%' appended)
        # e.g. 2000 (12.5%)

# tally up establishment count by rating (from rating frequency)
counts = Counter(e["RatingValue"] for e in gradable)
    # returns something like: {"5": 1200, "4": 400, "3": 150, "2": 50, "1": 10, "0": 5}.

# get totals by banding
critical = counts["0"] + counts["1"] + counts["2"]
fail = counts["3"] + counts["4"]
passed = counts["5"]

# output rating counts and percentage of total by class & banding
print("\nThree-tier:")
print(f"  Critical (0-2*): {pct(critical, total)}")
print(f"  Fail (3-4*):     {pct(fail, total)}")
print(f"  Pass (5*):       {pct(passed, total)}")

print("\nBinary (<5* = fail):")
print(f"  Fail (0-4*): {pct(critical + fail, total)}")
print(f"  Pass (5*):   {pct(passed, total)}")

print("\nBinary (<=2* = fail):")
print(f"  Fail (0-2*): {pct(critical, total)}")
print(f"  Pass (3-5*): {pct(fail + passed, total)}")

Gradable establishments: 71002

Three-tier:
  Critical (0-2*): 3973 (5.6%)
  Fail (3-4*):     18526 (26.1%)
  Pass (5*):       48503 (68.3%)

Binary (<5* = fail):
  Fail (0-4*): 22499 (31.7%)
  Pass (5*):   48503 (68.3%)

Binary (<=2* = fail):
  Fail (0-2*): 3973 (5.6%)
  Pass (3-5*): 67029 (94.4%)


-> The 'gradable' establishment count drops by c. 10k from the London total, to around 71k total.  

When looking 'London-wide', the proportion of 'critical fails' drops slighly when compared to Barking & Dagenham (5.6%, from 8.6%). This drop in representation shouldn't present an issue when building a regression model later, as the absolute count of c. 4,000 establishments in the 0-2* range should be plenty for a useable signal to be detected.  

Class weight balancing will still likely be needed, but I'm happy with my intended approach: looking at a 'three-tier' classification (5* = pass, 3-4* = fail, 0-2* = critical fail), with either of the two binary approaches (0-2* = fail, 0-4* = fail) as a fallback if initial model performance falls short.

### Save raw data pull

My next step is to save the pulled data, as it will not persist (for example) in the event of a kernel restart, which would require the data to be re-pulled. 

This will conclude the initial data exploration stage, with the cleaning stage to be completed in a subsequent notebook.

This also preserves a 'snapshot' of the data at a specific point in time, as logged below:

In [12]:
import datetime, json

with open("../data/raw/fsa_london_establishments.json", "w") as f:
    json.dump(all_establishments, f)

print(f"Saved {len(all_establishments)} records.")

metadata = {
    "pulled_at": datetime.datetime.now().isoformat(),
    "source": "FSA Food Hygiene Rating Scheme API",
    "record_count": len(all_establishments),
}

with open("../data/raw/fsa_london_establishments_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(metadata)

Saved 81217 records.
{'pulled_at': '2026-08-23T14:55:45.323364', 'source': 'FSA Food Hygiene Rating Scheme API', 'record_count': 81217}
